# 🧠📦 ThinkDome: 4-Bit Quantized LLM Sandbox Test

This notebook sets up and runs the **ThinkDome Autonomous LLM Sandbox & Tool Orchestrator** inside a Kaggle notebook environment, then tests the sandbox by loading and running a **4-bit quantized open-source LLM** entirely inside the sandboxed execution environment.

### What This Demonstrates
- ThinkDome SDK & API server bootstrap on Kaggle
- User registration, JWT session auth, and API key provisioning
- **Sandbox environment creation** with custom resource limits (RAM, CPU, timeout)
- **Sandbox selection** via `X-Sandbox-Id` header
- **4-bit quantized LLM inference** (Qwen2.5-1.5B-Instruct) executed inside the sandbox

### ⚠️ Kaggle Environment Notes
- Kaggle kernels run inside containers and **do not support Docker-in-Docker**, so ThinkDome uses the `subprocess` backend.
- Enable **GPU (T4)** in Kaggle kernel settings for CUDA-accelerated 4-bit inference.
- The API interface, role-based privilege checks, schemas, and toolsets remain identical to production.

## 🛠️ Step 1: Install Dependencies
Install ThinkDome from GitHub and the quantization libraries (bitsandbytes, transformers, accelerate).

In [ ]:
# Install ThinkDome from GitHub
!pip install -q git+https://github.com/abeldirectory252/ThinkDome.git

# Install 4-bit quantization dependencies
!pip install -q bitsandbytes accelerate transformers torch

print("✅ All dependencies installed.")

## 🐍 Step 2: Quick SDK Smoke Test
Verify ThinkDome's programmatic Sandbox SDK works before starting the server.

In [ ]:
from thinkdome import Sandbox

with Sandbox(backend="subprocess", timeout=10) as dome:
    result = dome.run("import sys; print(f'Python {sys.version} inside ThinkDome SDK')")
    print("Success:", result.success)
    print("Output:", result.output.strip())
    print("Duration (ms):", result.duration_ms)

## ⚙️ Step 3: Configure Environment & Start API Server

In [ ]:
import os, subprocess, time, sys

# Configure subprocess backend and super admin key
os.environ["EXECUTOR_BACKEND"] = "subprocess"
os.environ["API_KEY"] = "sk_tb_admin_super_secret_override_token"
os.environ["FILE_STORAGE_DIR"] = "./storage"
os.makedirs("./storage", exist_ok=True)

# Start ThinkDome API server in the background
print("Starting ThinkDome API server...")
server_process = subprocess.Popen(
    [sys.executable, "-m", "thinkdome.cli", "serve", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(5)

if server_process.poll() is None:
    print("✅ ThinkDome API server is running on http://127.0.0.1:8000")
else:
    print("❌ Server failed to start:")
    stdout, stderr = server_process.communicate()
    print(stderr)

## 🔍 Step 4: Health Check & API Verification

In [ ]:
import httpx

BASE_URL = "http://127.0.0.1:8000"

# Liveness check
resp = httpx.get(f"{BASE_URL}/health")
print("Health Check:", resp.status_code, resp.json())

# Schema check
schema_resp = httpx.get(f"{BASE_URL}/orchestrator_schema.json")
print("Orchestrator Schema tools:", len(schema_resp.json()), "tools loaded")

## 🔐 Step 5: Register User & Obtain Auth Tokens

In [ ]:
SUPER_KEY = os.environ.get("API_KEY")
admin_headers = {"Authorization": f"Bearer {SUPER_KEY}"}

# Register a user
reg = httpx.post(f"{BASE_URL}/v1/auth/register",
                 json={"username": "kaggle_operator", "password": "secure_password_123"})
print("Registration:", reg.status_code, reg.json())

# Login to get JWT
login = httpx.post(f"{BASE_URL}/v1/auth/login",
                   json={"username": "kaggle_operator", "password": "secure_password_123"})
session_data = login.json()
jwt_token = session_data.get("access_token")
print("🔑 JWT Token:", jwt_token[:40] + "...")

# Create API keys
llm_key = httpx.post(f"{BASE_URL}/v1/admin/keys", headers=admin_headers,
                     json={"display_name": "Kaggle LLM Agent", "token_type": "LLM"})
llm_token = llm_key.json()["token"]
print("🔑 LLM Token:", llm_token[:30] + "...")

admin_key = httpx.post(f"{BASE_URL}/v1/admin/keys", headers=admin_headers,
                       json={"display_name": "Kaggle Admin Operator", "token_type": "ADMIN"})
admin_token = admin_key.json()["token"]
print("🔑 Admin Token:", admin_token[:30] + "...")

## 📦 Step 6: Create Sandbox Environment
Provision a sandbox with enough resources for 4-bit LLM inference (4 GB RAM, 4 vCPU, 300s timeout, network enabled for model download).

In [ ]:
# Create a high-resource sandbox for LLM inference
sandbox_resp = httpx.post(
    f"{BASE_URL}/v1/admin/sandboxes",
    headers=admin_headers,
    json={
        "name": "LLM Inference Sandbox",
        "memory_mb": 4096,
        "cpu_cores": 4.0,
        "timeout_sec": 300,
        "network_enabled": True
    }
)
sandbox = sandbox_resp.json()
sandbox_id = sandbox["sandbox_id"]

print(f"✅ Sandbox created: {sandbox['name']}")
print(f"   ID:       {sandbox_id}")
print(f"   Status:   {sandbox['status']}")
print(f"   Memory:   {sandbox['memory_mb']} MB")
print(f"   CPU:      {sandbox['cpu_cores']} vCPU")
print(f"   Timeout:  {sandbox['timeout_sec']}s")
print(f"   Network:  {'Enabled' if sandbox['network_enabled'] else 'Disabled'}")
print(f"   Cost:     ${sandbox['cost_per_hour']:.3f}/hr")

## 🧪 Step 7: Verify Sandbox Selection Works
List all active sandboxes and confirm the orchestrator routes to the correct one via `X-Sandbox-Id` header.

In [ ]:
# List all sandboxes
sandboxes = httpx.get(f"{BASE_URL}/v1/admin/sandboxes", headers=admin_headers).json()
print(f"Total sandboxes: {len(sandboxes)}")
for sb in sandboxes:
    status_icon = '🟢' if sb['status'] == 'active' else '🔴'
    print(f"  {status_icon} {sb['name']} | {sb['memory_mb']}MB | {sb['cpu_cores']}vCPU | ID: {sb['sandbox_id']}")

# Test: run a simple command targeting our specific sandbox
test_resp = httpx.post(
    f"{BASE_URL}/v1/orchestrate",
    headers={
        "Authorization": f"Bearer {admin_token}",
        "X-Sandbox-Id": sandbox_id
    },
    json={
        "type": "tool_use",
        "id": "test_sandbox_selection",
        "name": "run_code",
        "input": {"code": "import platform; print(f'Running on {platform.system()} {platform.machine()}')"}
    }
)
print("\nSandbox execution test:", test_resp.json())

## 🤖 Step 8: Load 4-Bit Quantized LLM Inside the Sandbox
This is the core test — we execute code **inside the ThinkDome sandbox** that:
1. Loads `Qwen/Qwen2.5-1.5B-Instruct` with **4-bit quantization** via `bitsandbytes`
2. Runs inference to generate a response
3. Returns the output through the orchestrator API

The model is small enough for Kaggle's T4 GPU (16 GB VRAM). 4-bit quantization reduces memory from ~3 GB to ~1 GB.

In [ ]:
# The code that will run INSIDE the ThinkDome sandbox
LLM_CODE = '''
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print("[1/4] Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[2/4] Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"[3/4] Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Print model memory footprint
mem_bytes = model.get_memory_footprint()
print(f"    Model memory footprint: {mem_bytes / 1e9:.2f} GB (4-bit quantized)")

print("[4/4] Running inference...")
prompt = "Explain what a sandbox is in software engineering in 2 sentences."
messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\n" + "=" * 60)
print(f"PROMPT: {prompt}")
print(f"RESPONSE: {response}")
print("=" * 60)
print(f"\nDevice: {model.device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
'''

print(f"Submitting {len(LLM_CODE)} chars of LLM code to sandbox '{sandbox_id}'...")
print("This may take 1-3 minutes for model download and loading.\n")

In [ ]:
import json

# Execute the 4-bit LLM code inside ThinkDome sandbox
llm_resp = httpx.post(
    f"{BASE_URL}/v1/orchestrate",
    headers={
        "Authorization": f"Bearer {admin_token}",
        "X-Sandbox-Id": sandbox_id
    },
    json={
        "type": "tool_use",
        "id": "toolu_4bit_llm_inference",
        "name": "run_code",
        "input": {
            "code": LLM_CODE,
            "language": "python",
            "allow_network": True
        }
    },
    timeout=360.0  # 6 min timeout for model download + inference
)

result = llm_resp.json()

if result.get("is_error"):
    print("❌ Execution failed:")
    print(result["content"])
else:
    # Parse the nested JSON content
    try:
        output = json.loads(result["content"])
        print("✅ 4-Bit LLM Inference Complete!\n")
        print("── STDOUT ──")
        print(output["stdout"])
        if output.get("stderr"):
            print("── STDERR ──")
            print(output["stderr"][:500])
        print(f"\n⏱️  Duration: {output['duration_ms']:.0f}ms")
        print(f"🔄 Exit Code: {output['exit_code']}")
        print(f"⏰ Timed Out: {output['timed_out']}")
    except json.JSONDecodeError:
        print(result["content"])

## 🔬 Step 9: Test Privilege Enforcement
Verify that LLM tokens cannot access destructive tools, while Admin tokens can.

In [ ]:
def call_tool(name, inputs, token, sandbox=None):
    headers = {"Authorization": f"Bearer {token}"}
    if sandbox:
        headers["X-Sandbox-Id"] = sandbox
    resp = httpx.post(
        f"{BASE_URL}/v1/orchestrate",
        headers=headers,
        json={"type": "tool_use", "id": f"call_{name}", "name": name, "input": inputs}
    )
    return resp.json()

# Test 1: LLM token — memory_store (allowed)
r1 = call_tool("memory_store",
               {"key": "test_note", "content": "Running 4-bit LLM in sandbox", "tags": ["llm", "4bit"]},
               llm_token, sandbox_id)
print("✅ memory_store (LLM):", "Success" if not r1.get("is_error") else r1["content"])

# Test 2: LLM token — shell_exec (should be DENIED)
r2 = call_tool("shell_exec", {"command": "whoami"}, llm_token, sandbox_id)
print("🔒 shell_exec (LLM):", r2.get("content", "")[:80])

# Test 3: Admin token — shell_exec (allowed)
r3 = call_tool("shell_exec", {"command": "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU detected'"},
               admin_token, sandbox_id)
print("✅ shell_exec (Admin):", r3.get("content", "")[:120])

## 📊 Step 10: Check Execution Logs
Review the audit trail of all orchestrator executions.

In [ ]:
logs = httpx.get(f"{BASE_URL}/v1/admin/logs", headers=admin_headers).json()

print(f"Total execution logs: {len(logs)}\n")
print(f"{'Time':<12} {'Tool':<16} {'Status':<10} {'Latency':<12}")
print("-" * 50)
for log in logs[:10]:
    t = log['timestamp'].split('T')[1][:8]
    print(f"{t:<12} {log['tool_name']:<16} {log['status']:<10} {log['duration_ms']:.1f}ms")

## 🌐 Step 11: Expose Externally (Optional)
Use Localtunnel or Ngrok to access ThinkDome from outside Kaggle.

In [ ]:
# Install localtunnel
!npm install -g localtunnel

print("Localtunnel installed. Run in a separate terminal:")
print("lt --port 8000")

## 🛑 Step 12: Cleanup & Shutdown

In [ ]:
# Terminate the sandbox
del_resp = httpx.delete(
    f"{BASE_URL}/v1/admin/sandboxes/{sandbox_id}",
    headers=admin_headers
)
print(f"Sandbox terminated: {del_resp.status_code}")

# Stop the server
server_process.terminate()
print("✅ ThinkDome server stopped.")